# Simulated Annealing für das Bin-Packing Problem

## Problem und Ziel

In dieser Aufgabe wird das **Bin Packing Problem** gelöst. Es gibt eine Menge von Items mit festen Gewichten und beliebig viele Bins mit derselben Kapazität. Gesucht ist eine zulässige Zuordnung von Items zu Bins, bei der kein Bin seine Kapazität überschreitet. Gleichzeitig soll die Anzahl der verwendeten Bins möglichst klein werden. Die Zielfunktion ist deshalb sehr direkt: Je weniger Bins eine Lösung braucht, desto besser ist sie.

Das Problem ist ein kombinatorisches Optimierungsproblem. Schon kleine Änderungen an einer Zuordnung können viel ausmachen, weil ein einzelnes Item darüber entscheidet, ob ein Bin frei wird oder nicht.
Um für dieses NP-harte Problem eine Lösung zu finden wird eine Metaheuristik angewendet, welche in effiezienter Laufzeit eine Lösung finden kann, welche nahe am Optimum liegt.

## Konstruktive Heuristik

Als Startlösung verwenden wir die konstruktive Heuristik aus `ConstructiveHeuristics.py`. Neben der gegebenen `BinPerItem`, bei der jedes Item in einen eigenen Bin gelegt wird, ist `FAMAP` umgesetzt. `FAMAP` steht hier für `FitAsManyAsPossible`: Die Items werden in ihrer Eingabereihenfolge betrachtet. Solange das nächste Item noch in den aktuellen Bin passt, wird es dort abgelegt. Wenn die Kapazität überschritten würde, wird ein neuer Bin geöffnet.

Der Vorteil ist, dass diese Startlösung sehr schnell erzeugt wird und immer zulässig bleibt. Der Nachteil ist genauso klar: Die Reihenfolge der Items entscheidet stark über die Qualität. Ein späteres Item wird nicht noch einmal zurücksortiert. 

## Metaheuristik

Als Metaheuristik wird **Simulated Annealing** verwendet. Die Klassifizierung ist damit: trajektorienbasiert, stochastisch und akzeptanzbasiert. Es wird immer von einer aktuellen Lösung ausgegangen. Daraus wird eine Nachbarlösung erzeugt. Wenn sie besser oder gleich gut ist, wird sie angenommen. Wenn sie schlechter ist, kann sie trotzdem angenommen werden. Diese Wahrscheinlichkeit hängt von einer Temperatur ab. Beim Simlated Annealing wird diese Temperatur über mehrere Suchiterationen gesenkt, was "Abkühlung" genannt wird.

So entsteht am Anfang mehr Freiheit in der Suche. Die Methode darf auch einmal in eine schlechtere Richtung gehen, wenn dadurch später bessere Bereiche erreichbar werden. Mit sinkender Temperatur wird die Suche strenger und nimmt schlechtere Lösungen immer seltener an. Das ist der zentrale Wechsel zwischen Diversifizierung und Intensivierung.

## Suchoperatoren

Die Nachbarschaften liegen in `Neighbourhood.py` und werden in `ImprovementAlgorithm.py` über `neighborhoodTypes` ausgewählt. Dadurch muss das Notebook keine Nachbarschaftsobjekte selbst bauen. Es setzt nur die Parameter und startet den Algorithmus.

Wir haben zwei verschiedene Änsatze zur Nachbarschaftssuche implementiert, aus welchen zum Start des Annealing-Prozesses gewählt werden kann.

`RepackItems` nimmt ein Item und verschiebt es in einen anderen bereits vorhandenen Bin, falls dort genug Restkapazität vorhanden ist. Dieser Operator ist klein, schnell und erzeugt viele lokale Änderungen. Er eignet sich gut, um eine bestehende Packung fein zu verbessern.

`EmptyBin` verfolgt ein direkteres Ziel: Ein zufällig gewählter nicht-leerer Bin soll komplett geleert werden. Seine Items werden per Best-Fit in andere vorhandene Bins verteilt. Ein Move zählt nur, wenn der Quell-Bin vollständig verschwindet und keine Kapazität überschritten wird. Das passt gut zur Zielfunktion, weil ein erfolgreicher Move die Anzahl der Bins direkt senken kann.

## Diversifizierung und Intensivierung

Die Diversifizierung kommt aus zwei Quellen. Erstens werden Nachbarn zufällig erzeugt. Zweitens akzeptiert Simulated Annealing bei hoher Temperatur auch schlechtere Lösungen. Dadurch hängt die Suche nicht sofort an der ersten lokalen Verbesserung fest.

Die Intensivierung entsteht durch die Abkühlung. Wenn die Temperatur kleiner wird, wird die Suche vorsichtiger. Dann zählen vor allem Verbesserungen oder gleich gute Lösungen. Zusätzlich hilft der `EmptyBin`-Operator bei der Intensivierung, weil er gezielt versucht, eine Lösung kompakter zu machen.

## Terminierung und Parameter

Das Terminierungskriterium ist die Temperatur. Der Algorithmus läuft, bis `temperature` unter einen `threshold` fällt. Innerhalb einer Temperaturstufe begrenzt `maxMarkovLength`, wie viele Nachbarn betrachtet werden. `coolingSpeed` steuert, wie schnell abgekühlt wird. `numberOfMoves` bestimmt, wie viele mögliche Moves pro Nachbarschaft erzeugt werden.

Die Parameter werden im Notebook direkt gesetzt und optional per Grid Search verglichen. Für die Parameterwahl werden Qualität und Laufzeit gemeinsam betrachtet. Beste Qualität ist nicht automatisch die beste Wahl, wenn sie sehr viel Laufzeit kostet. Deshalb werden auch ausgewogene Scores und Laufzeit-Qualitäts-Verhältnisse berechnet.

## Grid Search

Der Grid Search dient dazu, mehrere Parameterkombinationen systematisch zu vergleichen. Aus Zeitgründen konnte er nicht auf allen Instanzen durchgeführt werden. Stattdessen läuft er nur auf dem kleinsten Datensatz. Außerdem wurden einige interessante Werte aus dem Grid weggelassen, damit die Anzahl der Simulated-Annealing-Läufe begrenzt bleibt. Die Ergebnisse sind deshalb keine vollständige Parameterstudie, geben aber eine brauchbare Orientierung.

Bewertet wird jede Einstellung mit mehreren Kennzahlen. `meanFinalBins` misst die mittlere Anzahl verwendeter Bins und steht damit für die Lösungsqualität. `meanRuntime` misst die mittlere Laufzeit. `runtimeQualityRatio` und `squaredRuntimeQualityRatio` setzen Laufzeit und Verbesserung ins Verhältnis. `balancedScore` normiert Qualität und Laufzeit und addiert beide Strafwerte. Zusätzlich wird die schnellste noch akzeptable Lösung betrachtet, wenn sie höchstens zwei Prozent schlechter als die beste Qualität ist.

Im Code werden daraus mehrere Empfehlungen ausgegeben: beste Qualität, schnellste Laufzeit, ausgewogene Variante und schnellste akzeptable Variante. Für die weitere Ausführung wird bewusst die ausgewogene Variante gesetzt, weil sie Qualität und Laufzeit gemeinsam berücksichtigt. Da die Laufzeiten abhängig von System sind, auf dem der Test läuft, varriieren auch die Bewertungen dementsprechend und können von unseren Empfehlungen abweichen.

Die erzeugten Grafiken zeigen die wichtigsten Vergleiche der Parameter und Metriken. Sie dienen vor allem dazu, die Tendenzen im Grid Search sichtbar zu machen, ohne jede Kombination einzeln im Text zu beschreiben.

## Pseudocode

```text
Lade alle Instanzen
Für jede Instanz:
    Erzeuge eine Startlösung mit FAMAP
    Setze currentSolution und bestSolution auf die Startlösung
    Solange die Temperatur größer als threshold ist:
        Erzeuge bis maxMarkovLength Nachbarlösungen
        Wähle die erste konfigurierte Nachbarschaft
        Erzeuge numberOfMoves Moves
        Wähle zufällig einen Move aus
        Akzeptiere bessere Lösungen immer
        Akzeptiere schlechtere Lösungen abhängig von Temperatur und Delta
        Aktualisiere bestSolution, wenn weniger Bins verwendet werden
        Kühle die Temperatur ab
    Prüfe die Machbarkeit
    Speichere die Lösung als CSV
```

## Ergebnisse und Schwierigkeiten

Die Ergebnisse werden in Abschnitt 5 gesammelt. Für jede Instanz werden Start-Bins, finale Bins, Machbarkeit und der Pfad zur CSV-Datei ausgegeben. Wichtig ist dabei nicht nur eine kleine Bin-Anzahl, sondern auch die Zulässigkeit. Jede Lösung wird deshalb noch einmal gegen die Kapazitäten geprüft, bevor sie gespeichert wird.

Eine Schwierigkeit war `EmptyBin`. Teilweise verschobene Bins sind für diese Nachbarschaft nicht sinnvoll, weil der Bin dann nicht wirklich eingespart wird. Deshalb ist der Operator jetzt ganz oder gar nicht: Nur wenn alle Items des Quell-Bins in andere Bins passen, wird der Move gespeichert. Das macht die Suche verlässlicher und passt besser zum Ziel, möglichst viele Bins loszuwerden.

Außerdem war das Finden guter Parameter für den Algorithmus schwierig, da der Einfluss des `coolingSpeed` und der `maxMarkovlength` schwierig zu bewerten war. 
Deswegen wurde ein Grid-Search Ansatz gewählt, um bessere Parameter zu finden.

Am Ende ist die Lösung bewusst praktisch aufgebaut. Die konstruktive Heuristik liefert schnell einen Start. Simulated Annealing verbessert diesen Start mit kontrollierter Zufälligkeit. Die Auswertung zeigt direkt, ob die Lösung machbar ist und wie viele Bins am Ende übrig bleiben. 

# 1. Imports und globale Einstellungen

In [17]:
import os

os.makedirs('/tmp/matplotlib', exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

try:
    import matplotlib
except ImportError:
    print('matplotlib ist nicht installiert; Grid-Search-Plots werden übersprungen.')

In [18]:
from __future__ import annotations

import csv
import time

import numpy as np

from Solver import *

In [ ]:
START_HEURISTIC = 'FAMAP' # 'BPI' oder 'FAMAP'

SA_PARAMETERS = {
    'neighborhoodTypes': ['EmptyBin'], #['RepackItems'] oder ['EmptyBin']
    'temperature': 0.95,
    'coolingSpeed': 0.5,
    'threshold': 1e-2,
    'maxMarkovLength': 200,
    'numberOfMoves': 100,
}

NOTEBOOK_TEST_MODE = __name__ == '__notebook_test__'
seed = 2


# 2. Daten laden

In [12]:
files = Files()

try:
    paths = [] if NOTEBOOK_TEST_MODE else sorted(files.GetFiles())
except FileNotFoundError:
    paths = []
    print("Kein Data-Ordner gefunden. Lege ../Data an oder passe InputData.Files an.")

fileNames = [path.split('/')[-1] for path in paths]
print(f"Alle Dateien im Zielordner sind: {fileNames} \n")

dataSets = []
for path in paths:
    print("________________________________________________________________________________________")
    print(f"Lade Instanz: {path.split('/')[-1]}")

    # InputData.DataLoad validiert die JSON-Datei und baut DataItem-Objekte und DataBinCapacity auf.
    codeDirectory = os.path.dirname(os.path.abspath(Files.__init__.__code__.co_filename))
    inputPath = path if os.path.isabs(path) else os.path.abspath(os.path.join(codeDirectory, path))
    data = InputData(inputPath)
    dataSets.append(data)

allDataSets = list(dataSets)

Alle Dateien im Zielordner sind: ['..\\Data\\Falkenauer_u1000_13.json', '..\\Data\\Falkenauer_u120_09.json', '..\\Data\\Falkenauer_u500_05.json', '..\\Data\\csBA500_12.json', '..\\Data\\csBB250_13.json'] 

________________________________________________________________________________________
Lade Instanz: ..\Data\Falkenauer_u1000_13.json
Number of items: 1000
Bincapacity: 150

________________________________________________________________________________________
Lade Instanz: ..\Data\Falkenauer_u120_09.json
Number of items: 120
Bincapacity: 150

________________________________________________________________________________________
Lade Instanz: ..\Data\Falkenauer_u500_05.json
Number of items: 500
Bincapacity: 150

________________________________________________________________________________________
Lade Instanz: ..\Data\csBA500_12.json
Number of items: 5220
Bincapacity: 1500000

________________________________________________________________________________________
Lade Insta

# 3. Konstruktive Startlösungen

In [ ]:
constructiveResults = []
for data in dataSets:
    print("________________________________________________________________________________________")
    print(f"Konstruktive Phase für: {data.filename}")

    startTime = time.time()
    solver = Solver(data)
    startSolution = solver.ConstructionPhase(START_HEURISTIC)

    startSolution.FeasibilityCheckOutput(data)
    #print(startSolution.Bins)

    runtime = time.time() - startTime
    constructiveResults.append({
        'path': data.path,
        'instance': data.filename,
        'data': data,
        'constructiveHeuristic': START_HEURISTIC,
        'startSolution': startSolution,
        'constructiveRuntime': runtime,
    })

________________________________________________________________________________________
Konstruktive Phase für: Falkenauer_u1000_13.json
Generating an initial solution according to FAMAP.
Constructive solution found: The number of bins is 554. 

The allocation is feasible! All bins remain within their capacity.
Maximum weight in a bin: 150
Minimum weight in a bin: 20


# 4. Simulated Annealing

In [14]:
# Simulated Annealing ist als Unterklasse von ImprovementAlgorithm in ImprovementAlgorithm.py umgesetzt.
# Die Nachbarschaft wird über SA_PARAMETERS['neighborhoodTypes'] ausgewählt und im Algorithmus erzeugt.
# Solver.py ruft algorithm.Initialize(...) und algorithm.Run(startSolution) auf.

finalResults = []

for result in constructiveResults:
    data = result['data']
    startSolution = result['startSolution']
   
    algorithm = SimulatedAnnealing(
        inputData=data,
        **SA_PARAMETERS,
    )
    solver = Solver(data)
    startTime = time.time()
    finalSolution = solver.Run(result['constructiveHeuristic'], algorithm)
    runtime = time.time() - startTime
    print(f"Number of bins: {finalSolution.NumberOfBins}")

    result['finalSolution'] = finalSolution
    result['improvementRuntime'] = runtime
    finalResults.append(result)

Generating an initial solution according to FAMAP.
Constructive solution found: The number of bins is 554. 

Best found Solution: The number of bins is 406.

The allocation is feasible! All bins remain within their capacity.
Maximum weight in a bin: 150
Minimum weight in a bin: 131
Number of bins: 406


# 5. Output und Ergebnisübersicht

In [15]:
def calculate_bin_weights(solution, data):
    binWeights = {binId: 0 for binId in sorted(set(solution.Allocation.values()))}

    for itemId, binId in solution.Allocation.items():
        binWeights[binId] += data.InputItems[itemId].weight

    return binWeights


def is_feasible(solution, data):
    binWeights = calculate_bin_weights(solution, data)
    capacity = data.InputBinCapacity.capacity
    return all(weight <= capacity for weight in binWeights.values())


def write_solution_csv(solution, data, outputFolder='../Solutions'):
    os.makedirs(outputFolder, exist_ok=True)

    instanceName = os.path.splitext(data.filename)[0]
    outputPath = os.path.join(outputFolder, f'Solution-{instanceName}.csv')
    binIdMap = {binId: index for index, binId in enumerate(sorted(set(solution.Allocation.values())))}

    with open(outputPath, 'w', newline='') as outputFile:
        writer = csv.writer(outputFile)
        writer.writerow(['itemId', 'binId'])

        for itemId in sorted(solution.Allocation):
            writer.writerow([itemId, binIdMap[solution.Allocation[itemId]]])

    return outputPath

results = []

for result in finalResults:
    data = result['data']
    finalSolution = result['finalSolution']
    feasible = finalSolution.FeasibilityCheck(data)
    #feasible = is_feasible(finalSolution, data)
    solutionPath = write_solution_csv(finalSolution, data) if feasible else None

    if solutionPath is not None:
        print(f"Lösung gespeichert: {solutionPath}")
    else:
        print(f"Keine CSV geschrieben, da die Lösung für {result['instance']} nicht zulässig ist.")

    results.append({
        'Instanz': result['instance'],
        'Konstruktionsheuristik': result['constructiveHeuristic'],
        'StartBins': result['startSolution'].NumberOfBins,
        'FinalBins': finalSolution.NumberOfBins,
        'Konstruktionszeit': round(result['constructiveRuntime'], 4),
        'Verbesserungszeit': round(result['improvementRuntime'], 4),
        'Erlaubt': feasible,
        'SolutionFile': solutionPath,
    })

columns = ['Instanz', 'Konstruktionsheuristik', 'StartBins', 'FinalBins', 'Konstruktionszeit','Verbesserungszeit', 'Erlaubt', 'SolutionFile']
columnWidths = {
    column: max([len(column), *(len(str(row[column])) for row in results)])
    for column in columns
}

header = ' | '.join(column.ljust(columnWidths[column]) for column in columns)
separator = '-+-'.join('-' * columnWidths[column] for column in columns)
print(header)
print(separator)

for row in results:
    print(' | '.join(str(row[column]).ljust(columnWidths[column]) for column in columns))

results

Lösung gespeichert: ../Solutions\Solution-Falkenauer_u1000_13.csv
Instanz                  | Konstruktionsheuristik | StartBins | FinalBins | Konstruktionszeit | Verbesserungszeit | Erlaubt | SolutionFile                                 
-------------------------+------------------------+-----------+-----------+-------------------+-------------------+---------+----------------------------------------------
Falkenauer_u1000_13.json | FAMAP                  | 554       | 406       | 0.0011            | 13.7107           | True    | ../Solutions\Solution-Falkenauer_u1000_13.csv


[{'Instanz': 'Falkenauer_u1000_13.json',
  'Konstruktionsheuristik': 'FAMAP',
  'StartBins': 554,
  'FinalBins': 406,
  'Konstruktionszeit': 0.0011,
  'Verbesserungszeit': 13.7107,
  'Erlaubt': True,
  'SolutionFile': '../Solutions\\Solution-Falkenauer_u1000_13.csv'}]

## Optionaler Grid Search zur Parameterwahl

Dieser Block dient dazu, die Parameter von Simulated Annealing systematisch zu vergleichen, statt sie nur manuell zu setzen. Dafür werden mehrere Kombinationen aus Starttemperatur und Abkühlgeschwindigkeit getestet und anhand von Lösungsqualität und Laufzeit bewertet. Die Auswertung erzeugt Empfehlungen für unterschiedliche Ziele: beste Qualität, kürzeste Laufzeit, ausgewogene Laufzeit-Qualitäts-Balance und schnellste noch akzeptable Lösung.

Der Grid Search ist standardmäßig deaktiviert, weil er je nach Instanzgröße viele SA-Läufe ausführt. Besonders Laufzeiten und daraus abgeleitete Scores sind hardwareabhängig: Auf einem anderen Rechner können absolute Zeiten und damit auch Laufzeit-Qualitäts-Verhältnisse anders ausfallen, während die reine Lösungsqualität besser vergleichbar bleibt.


In [ ]:
from copy import deepcopy
from itertools import product

try:
    from sklearn.model_selection import KFold, ParameterGrid
except ImportError:
    KFold = None
    ParameterGrid = None

DO_OPTIMIZATION = False #Set this to True to run Grid-Search
GRID_SEARCH_OUTPUT_FOLDER = '../Figures'
USED_BINS_AXIS_LIMITS = (40, 50)

SA_PARAMETER_DEFAULTS = {
    'threshold': 1e-2,
}

GRID_NEIGHBORHOOD_TYPES = [
    ['RepackItems'],
    ['EmptyBin'],
]

SA_PARAMETER_GRID = {
    'temperature': [0.75, 0.95],
    'coolingSpeed': [0.5, 0.75, 0.9],
    'neighborhoodTypes': GRID_NEIGHBORHOOD_TYPES,
    'maxMarkovLength': [200, 500, 1000],
    'numberOfMoves': [10, 50, 100],
}

GRID_SEARCH_PARAMETER_KEYS = list(SA_PARAMETER_GRID)


def make_sa_parameters(parameters):
    return {**SA_PARAMETER_DEFAULTS, **parameters}


def iter_parameter_grid(parameterGrid):
    if ParameterGrid is not None:
        yield from ParameterGrid(parameterGrid)
        return

    keys = list(parameterGrid)
    for values in product(*(parameterGrid[key] for key in keys)):
        yield dict(zip(keys, values))


def make_folds(items, n_splits=2):
    indices = np.arange(len(items))

    if len(indices) < 2:
        return [(indices, indices)]

    if KFold is not None:
        splitter = KFold(n_splits=min(n_splits, len(indices)), shuffle=True, random_state=seed)
        return list(splitter.split(indices))

    shuffled = np.random.default_rng(seed).permutation(indices)
    splitPoint = max(1, len(shuffled) // 2)
    return [
        (shuffled[:splitPoint], shuffled[splitPoint:]),
        (shuffled[splitPoint:], shuffled[:splitPoint]),
    ]


def make_constructive_result(data):
    startTime = time.time()
    solver = Solver(data)
    startSolution = solver.ConstructionPhase(START_HEURISTIC)
    runtime = time.time() - startTime

    return {
        'path': data.path,
        'instance': data.filename,
        'data': data,
        'heuristic': START_HEURISTIC,
        'startSolution': startSolution,
        'constructiveRuntime': runtime,
    }


def make_grid_search_constructive_results(sourceDataSets):
    if not sourceDataSets:
        return []

    smallestData = min(sourceDataSets, key=lambda data: len(data.InputItems))
    existingResult = next(
        (result for result in constructiveResults if result['data'] is smallestData),
        None,
    )

    if existingResult is not None:
        return [existingResult]

    return [make_constructive_result(smallestData)]


def run_sa_once(result, parameters, runSeed):
    data = result['data']
    evaluationLogic = EvaluationLogic(data)
    solutionPool = SolutionPool()
    startSolution = deepcopy(result['startSolution'])
    solutionPool.AddSolution(startSolution)

    algorithm = SimulatedAnnealing(inputData=data, **make_sa_parameters(parameters))
    algorithm.Initialize(evaluationLogic, solutionPool, np.random.default_rng(runSeed))

    startTime = time.time()
    finalSolution = algorithm.Run(startSolution)
    runtime = time.time() - startTime

    evaluationLogic.CalculateNumberOfBins(finalSolution)
    return {
        'instance': result['instance'],
        'startBins': result['startSolution'].NumberOfBins,
        'finalBins': finalSolution.NumberOfBins,
        'improvement': result['startSolution'].NumberOfBins - finalSolution.NumberOfBins,
        'runtime': runtime,
    }


def add_grid_search_scores(rows):
    if not rows:
        return rows

    bestBins = min(row['meanFinalBins'] for row in rows)
    fastestRuntime = min(row['meanRuntime'] for row in rows)
    slowestRuntime = max(row['meanRuntime'] for row in rows)
    worstBins = max(row['meanFinalBins'] for row in rows)
    runtimeRange = max(slowestRuntime - fastestRuntime, 1e-12)
    qualityRange = max(worstBins - bestBins, 1e-12)

    for row in rows:
        qualityPenalty = (row['meanFinalBins'] - bestBins) / qualityRange
        runtimePenalty = (row['meanRuntime'] - fastestRuntime) / runtimeRange
        row['runtimeQualityRatio'] = row['meanRuntime'] / max(row['meanImprovement'], 1e-9)
        row['squaredRuntimeQualityRatio'] = (row['meanRuntime'] ** 2) / max(row['meanImprovement'], 1e-9)
        row['balancedScore'] = qualityPenalty + runtimePenalty

    sortedByQuality = sorted(rows, key=lambda row: (row['meanFinalBins'], row['meanRuntime']))
    bestQuality = sortedByQuality[0]
    acceptableQualityLimit = bestQuality['meanFinalBins'] * 1.02
    acceptableRows = [row for row in rows if row['meanFinalBins'] <= acceptableQualityLimit]

    return {
        'allResults': rows,
        'bestQuality': bestQuality,
        'bestRuntime': min(rows, key=lambda row: row['meanRuntime']),
        'bestBalanced': min(rows, key=lambda row: row['balancedScore']),
        'bestRuntimeQualityRatio': min(rows, key=lambda row: row['runtimeQualityRatio']),
        'bestSquaredRuntimeQualityRatio': min(rows, key=lambda row: row['squaredRuntimeQualityRatio']),
        'fastestAcceptable': min(acceptableRows, key=lambda row: row['meanRuntime']),
    }


def plot_grid_search_results(gridSearchResults, outputFolder):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('matplotlib ist nicht installiert; Grid-Search-Plots werden übersprungen.')
        return []

    os.makedirs(outputFolder, exist_ok=True)
    rows = gridSearchResults['allResults']
    plotPaths = []

    def parameter_label(row):
        neighborhoodLabel = '+'.join(row['neighborhoodTypes'])
        return f"T={row['temperature']}, C={row['coolingSpeed']}, M={row['maxMarkovLength']}, Moves={row['numberOfMoves']}, N={neighborhoodLabel}"

    def plot_bar_chart(filename, metricKey, title, yLabel):
        sortedRows = sorted(rows, key=lambda row: row[metricKey])
        labels = [parameter_label(row) for row in sortedRows]
        values = [row[metricKey] for row in sortedRows]
        xPositions = range(len(sortedRows))
        figureWidth = max(9, len(sortedRows) * 0.75)

        path = os.path.join(outputFolder, filename)
        plt.figure(figsize=(figureWidth, 4.5))
        plt.bar(xPositions, values)
        plt.xticks(xPositions, labels, rotation=45, ha='right')
        plt.xlabel('Gridsearch-Parameterpaar')
        plt.ylabel(yLabel)
        if metricKey == 'meanFinalBins':
            plt.ylim(*USED_BINS_AXIS_LIMITS)
        plt.title(title)
        plt.tight_layout()
        plt.savefig(path, dpi=150)
        plt.close()
        plotPaths.append(path)

    def parameter_value_label(value):
        if isinstance(value, list):
            return '+'.join(value)
        return str(value)

    def parameter_sort_key(value):
        if isinstance(value, list):
            return tuple(value)
        return value

    def grouped_metric(rowsForValue, metricKey):
        return float(np.mean([row[metricKey] for row in rowsForValue]))

    def plot_parameter_impact(parameterKey):
        groupedRows = {}

        for row in rows:
            parameterValue = row[parameterKey]
            groupingKey = tuple(parameterValue) if isinstance(parameterValue, list) else parameterValue
            groupedRows.setdefault(groupingKey, []).append(row)

        sortedGroups = sorted(
            groupedRows.items(),
            key=lambda group: parameter_sort_key(list(group[0]) if isinstance(group[0], tuple) else group[0]),
        )
        labels = [parameter_value_label(list(groupKey) if isinstance(groupKey, tuple) else groupKey) for groupKey, _ in sortedGroups]
        metrics = [
            ('meanFinalBins', 'used_bins'),
            ('meanRuntime', 'Laufzeit (s)'),
            ('balancedScore', 'Balanced Score'),
        ]

        path = os.path.join(outputFolder, f'gridsearch-parameter-{parameterKey}.png')
        figure, axes = plt.subplots(1, len(metrics), figsize=(max(10, len(labels) * 1.4), 3.8))

        for axis, (metricKey, yLabel) in zip(axes, metrics):
            values = [grouped_metric(groupRows, metricKey) for _, groupRows in sortedGroups]
            axis.bar(range(len(labels)), values)
            axis.set_xticks(range(len(labels)))
            axis.set_xticklabels(labels, rotation=45, ha='right')
            axis.set_ylabel(yLabel)
            if metricKey == 'meanFinalBins':
                axis.set_ylim(*USED_BINS_AXIS_LIMITS)

        figure.suptitle(f'Gridsearch: Einfluss von {parameterKey}')
        figure.tight_layout()
        figure.savefig(path, dpi=150)
        plt.close(figure)
        plotPaths.append(path)

    def plot_parameter_heatmap(xKey, yKey, metricKey, filename, title, colorbarLabel):
        xValues = sorted({row[xKey] for row in rows})
        yValues = sorted({row[yKey] for row in rows})
        heatmapValues = np.full((len(yValues), len(xValues)), np.nan)

        for yIndex, yValue in enumerate(yValues):
            for xIndex, xValue in enumerate(xValues):
                matchingRows = [row for row in rows if row[xKey] == xValue and row[yKey] == yValue]
                if matchingRows:
                    heatmapValues[yIndex, xIndex] = grouped_metric(matchingRows, metricKey)

        path = os.path.join(outputFolder, filename)
        figure, axis = plt.subplots(figsize=(max(6, len(xValues) * 1.2), max(4, len(yValues) * 0.9)))
        imageLimits = {'vmin': USED_BINS_AXIS_LIMITS[0], 'vmax': USED_BINS_AXIS_LIMITS[1]} if metricKey == 'meanFinalBins' else {}
        image = axis.imshow(heatmapValues, aspect='auto', **imageLimits)
        axis.set_xticks(range(len(xValues)))
        axis.set_xticklabels([parameter_value_label(value) for value in xValues], rotation=45, ha='right')
        axis.set_yticks(range(len(yValues)))
        axis.set_yticklabels([parameter_value_label(value) for value in yValues])
        axis.set_xlabel(xKey)
        axis.set_ylabel(yKey)
        axis.set_title(title)
        figure.colorbar(image, ax=axis, label=colorbarLabel)
        figure.tight_layout()
        figure.savefig(path, dpi=150)
        plt.close(figure)
        plotPaths.append(path)

    for row in rows:
        row['runtimeRelativeToQuality'] = row['meanRuntime'] * row['meanFinalBins']

    plotDefinitions = [
        ('gridsearch-runtime.png', 'meanRuntime', 'Gridsearch sortiert nach Laufzeit', 'Mittlere Laufzeit (s)'),
        ('gridsearch-quality-used-bins.png', 'meanFinalBins', 'Gridsearch sortiert nach Ergebnisqualität', 'Mittlere used_bins'),
        ('gridsearch-runtime-relative-quality.png', 'runtimeRelativeToQuality', 'Gridsearch sortiert nach Laufzeit relativ zur Qualität', 'Mittlere Laufzeit × used_bins'),
    ]

    for filename, metricKey, title, yLabel in plotDefinitions:
        plot_bar_chart(filename, metricKey, title, yLabel)

    for parameterKey in GRID_SEARCH_PARAMETER_KEYS:
        plot_parameter_impact(parameterKey)

    plot_parameter_heatmap(
        'temperature',
        'coolingSpeed',
        'meanFinalBins',
        'gridsearch-heatmap-temperature-cooling-quality.png',
        'used_bins nach Temperatur und Abkühlgeschwindigkeit',
        'Mittlere used_bins',
    )
    plot_parameter_heatmap(
        'maxMarkovLength',
        'numberOfMoves',
        'balancedScore',
        'gridsearch-heatmap-markov-moves-balanced.png',
        'Balanced Score nach Markov-Länge und Moves',
        'Balanced Score',
    )

    return plotPaths


gridSearchResults = None

if DO_OPTIMIZATION:
    rawGridSearchResults = []
    gridSearchConstructiveResults = make_grid_search_constructive_results(allDataSets)
    folds = make_folds(gridSearchConstructiveResults, n_splits=2)
    parameterSettings = list(iter_parameter_grid(SA_PARAMETER_GRID))

    print(f'Starte Grid Search mit {len(parameterSettings)} Settings auf {len(gridSearchConstructiveResults)} Instanz(en).')

    for parameterIndex, parameters in enumerate(parameterSettings, start=1):
        foldMetrics = []
        print(f"\nGrid Search {parameterIndex}/{len(parameterSettings)}: {parameters}")

        for foldIndex, (_, validationIndices) in enumerate(folds, start=1):
            for validationIndex in validationIndices:
                runSeed = seed + parameterIndex * 1000 + foldIndex * 100 + int(validationIndex)
                metric = run_sa_once(gridSearchConstructiveResults[int(validationIndex)], parameters, runSeed)
                foldMetrics.append(metric)
                print(
                    f"  Fold {foldIndex}, Instanz {metric['instance']}: "
                    f"StartBins={metric['startBins']}, "
                    f"used_bins={metric['finalBins']}, "
                    f"Verbesserung={metric['improvement']}, "
                    f"Laufzeit={metric['runtime']:.2f}s"
                )

        resultRow = {
            **parameters,
            'meanFinalBins': float(np.mean([metric['finalBins'] for metric in foldMetrics])),
            'meanImprovement': float(np.mean([metric['improvement'] for metric in foldMetrics])),
            'meanRuntime': float(np.mean([metric['runtime'] for metric in foldMetrics])),
            'foldMetrics': foldMetrics,
        }
        rawGridSearchResults.append(resultRow)
        print(
            f"  Mittelwerte: used_bins={resultRow['meanFinalBins']:.2f}, "
            f"Verbesserung={resultRow['meanImprovement']:.2f}, "
            f"Laufzeit={resultRow['meanRuntime']:.2f}s"
        )

    gridSearchResults = add_grid_search_scores(rawGridSearchResults)
    gridSearchPlotPaths = plot_grid_search_results(gridSearchResults, GRID_SEARCH_OUTPUT_FOLDER)

    def best_parameters(resultKey):
        return make_sa_parameters({
            key: gridSearchResults[resultKey][key]
            for key in GRID_SEARCH_PARAMETER_KEYS
        })

    BEST_QUALITY_PARAMETERS = best_parameters('bestQuality')
    BEST_RUNTIME_PARAMETERS = best_parameters('bestRuntime')
    BEST_BALANCED_PARAMETERS = best_parameters('bestBalanced')
    BEST_FAST_ACCEPTABLE_PARAMETERS = best_parameters('fastestAcceptable')

    SA_PARAMETERS = BEST_BALANCED_PARAMETERS

    print('Beste Qualitätsparameter:', BEST_QUALITY_PARAMETERS)
    print('Schnellste Parameter:', BEST_RUNTIME_PARAMETERS)
    print('Ausgewogene Parameter:', BEST_BALANCED_PARAMETERS)
    print('Schnellste akzeptable Parameter:', BEST_FAST_ACCEPTABLE_PARAMETERS)
    print('Gespeicherte Plots:', gridSearchPlotPaths)
else:
    print('Grid Search ist deaktiviert. Setze DO_OPTIMIZATION = True, um SA_PARAMETERS automatisch zu kalibrieren.')

Generating an initial solution according to FAMAP.
Constructive solution found: The number of bins is 64. 

Starte Grid Search mit 108 Settings auf 1 Instanz(en).

Grid Search 1/108: {'coolingSpeed': 0.5, 'maxMarkovLength': 200, 'neighborhoodTypes': ['RepackItems'], 'numberOfMoves': 10, 'temperature': 0.75}
  Fold 1, Instanz Falkenauer_u120_09.json: StartBins=64, used_bins=50, Verbesserung=14, Laufzeit=0.83s
  Mittelwerte: used_bins=50.00, Verbesserung=14.00, Laufzeit=0.83s

Grid Search 2/108: {'coolingSpeed': 0.5, 'maxMarkovLength': 200, 'neighborhoodTypes': ['RepackItems'], 'numberOfMoves': 10, 'temperature': 0.95}
  Fold 1, Instanz Falkenauer_u120_09.json: StartBins=64, used_bins=50, Verbesserung=14, Laufzeit=0.84s
  Mittelwerte: used_bins=50.00, Verbesserung=14.00, Laufzeit=0.84s

Grid Search 3/108: {'coolingSpeed': 0.5, 'maxMarkovLength': 200, 'neighborhoodTypes': ['RepackItems'], 'numberOfMoves': 50, 'temperature': 0.75}
  Fold 1, Instanz Falkenauer_u120_09.json: StartBins=64, us